In [1]:
%pip install -qqq datasets==3.2.0 transformers==4.47.1 trl==0.14.0 peft==0.14.0 accelerate==1.2.1 bitsandbytes==0.45.2 wandb==0.19.7 tf-keras --progress-bar off

Note: you may need to restart the kernel to use updated packages.


In [2]:
import torch
from datasets import load_dataset
from peft import LoraConfig, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import GRPOConfig, GRPOTrainer

# Wandb Authorization

In [ ]:
# from google.colab import userdata
# import os
# import wandb

# # Retrieve your stored W&B API key
# wandb_key = userdata.get('wandb_key')  # This must match the key name you used to store it

# # Set the environment variable so wandb uses it
# os.environ['WANDB_API_KEY'] = wandb_key

# # Login without prompt
# wandb.login()


wandb: WARNING Calling wandb.login() after wandb.init() has no effect.


True

# Load the dataset

In [3]:
dataset = load_dataset("mlabonne/smoltldr")
print(dataset)

README.md:   0%|          | 0.00/981 [00:00<?, ?B/s]

c:\Users\Fardin Saad\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:144: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Fardin Saad\.cache\huggingface\hub\datasets--mlabonne--smoltldr. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. F

train-00000-of-00001.parquet:   0%|          | 0.00/1.44M [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


validation-00000-of-00001.parquet:   0%|          | 0.00/151k [00:00<?, ?B/s]

...

test-00000-of-00001.parquet:   0%|          | 0.00/151k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/200 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/200 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['prompt', 'completion'],
        num_rows: 2000
    })
    validation: Dataset({
        features: ['prompt', 'completion'],
        num_rows: 200
    })
    test: Dataset({
        features: ['prompt', 'completion'],
        num_rows: 200
    })
})


# Model

In [6]:
%pip install -qqq hf_xet

Note: you may need to restart the kernel to use updated packages.


In [7]:
model_id = "HuggingFaceTB/SmolLM-135M-Instruct"

# Check if CUDA is available
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

if torch.cuda.is_available():
    # GPU available - use quantization
    from transformers import BitsAndBytesConfig
    
    quantization_config = BitsAndBytesConfig(
        load_in_8bit=True,
        llm_int8_threshold=6.0,
        llm_int8_has_fp16_weight=False,
    )
    
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=quantization_config,
        torch_dtype=torch.float16,
        device_map="auto"
    )
else:
    # CPU only - no quantization
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=torch.float32,  # Use float32 for CPU
        device_map="cpu"
    )

tokenizer = AutoTokenizer.from_pretrained(model_id)

Using device: cpu


## Load LoRA

In [9]:
# Load LoRA
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules="all-linear",  # Apply to all linear layers
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 2,442,240 || all params: 136,957,248 || trainable%: 1.7832


## Reward function

In [ ]:
# Reward function
ideal_length = 50


def reward_len(completions, **kwargs):
    return [-abs(ideal_length - len(completion)) for completion in completions]

## Training arguments

In [ ]:
# Training arguments
training_args = GRPOConfig(
    output_dir="./grpo-output",
    learning_rate=2e-5,
    per_device_train_batch_size=2,     # ✅ small batch size
    gradient_accumulation_steps=1,
    max_prompt_length=128,             # ✅ shorter context
    max_completion_length=64,
    num_generations=4,                 # ✅ less memory than 8
    num_train_epochs=1,
    optim="adamw_torch" if not torch.cuda.is_available() else "adamw_8bit",  # Use regular optimizer for CPU
    bf16=False,
    fp16=torch.cuda.is_available(),    # Only use fp16 if GPU is available
    report_to=[],
    remove_unused_columns=False,
    logging_steps=1
)

## Trainer

In [ ]:
# Trainer
trainer = GRPOTrainer(
    model=model,
    reward_funcs=[reward_len],
    args=training_args,
    train_dataset=dataset["train"],
)

# Train model
# wandb.init(project="GRPO")
trainer.train()

Step,Training Loss
1,-0.000000
2,0.000500
3,0.000500
4,0.000500
5,0.000500
6,0.000700
7,0.000600
8,0.000600
9,0.000500
10,0.000500


In [8]:
# Check Hugging Face authentication
from huggingface_hub import HfFolder

token = HfFolder.get_token()
if token:
    # Display token safely (masked for security)
    masked_token = token[:8] + "..." + token[-8:] if len(token) > 16 else "***masked***"
    print(f"✅ Authenticated with HF token: {masked_token}")
    print(f"Token length: {len(token)} characters")
else:
    print("❌ No Hugging Face token found")
    
# Also check whoami
import subprocess
try:
    result = subprocess.run(['huggingface-cli', 'whoami'], 
                          capture_output=True, text=True, check=True)
    print(f"🔍 Logged in as: {result.stdout.strip()}")
except subprocess.CalledProcessError:
    print("❌ Not logged in via HF CLI")

✅ Authenticated with HF token: hf_fNJJM...knnTZQhT
Token length: 37 characters
🔍 Logged in as: Fardin19
🔍 Logged in as: Fardin19
